# 18 — A virada do 2º turno 2026: resultado, swing, quem virou, padrão e fluxo

**Escopo**: Presidente 2026, 2º turno (25/10/2026, totalização final do TSE) comparado ao 1º turno (snapshot final de 05/10/2026). Tudo calculado a partir dos Parquet de `data/processed/`.

**Definição da margem (D-039)**: nos dois turnos, margem = (PT − PL) em % dos votos **dos dois finalistas** (PT + PL), não em % dos válidos. Com os válidos no denominador do 1º turno, os candidatos que ficaram de fora inflariam mecanicamente a margem do 2º turno nos redutos de cada lado. Números de outras fontes que usem % dos válidos podem diferir. **Swing** = margem do 2º turno − margem do 1º (positivo = andou para o PT).

**Recorte**: o exterior (`zz`) fica fora dos mapas, do swing e do fluxo (D-027); os totais oficiais aparecem com e sem ele. Os dados são agregados por município (ecológicos): nada aqui é afirmação sobre eleitores individuais.

In [ ]:
import sys, pathlib

sys.path.insert(0, str(pathlib.Path("..") / "src"))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from eleicao import config, inferencia_ecologica as ie, projecao_t2 as pj
from eleicao.analise import carga, fluxo, graficos as g, regioes, virada
from eleicao.cores import carregar_paleta
from eleicao.segundo_turno_web import VOTOS_APOS_17H_IMPRENSA

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
paleta = carregar_paleta()
COR_PL, COR_PT = paleta[22], paleta[13]

t1 = virada.base_turno(carga.candidatos_2026(), carga.totais_2026())
t2 = virada.base_turno(carga.candidatos_t2_2026(), carga.totais_t2_2026())
d = virada.virada_municipal(t1, t2)  # Brasil sem exterior, 1 linha por município
print("municípios pareados (Brasil sem exterior):", len(d))

## 1. Resultado e margem nos dois turnos

Primeiro o total **oficial** (com o exterior), depois o recorte Brasil sem exterior, que é o que alimenta o swing e os mapas.

In [ ]:
cb, tb = carga.candidatos_t2_2026_br(), carga.totais_t2_2026_br().iloc[0]
v = cb.set_index(cb["nr_candidato"].astype(int))["votos"]
pt_of, pl_of = int(v[13]), int(v[22])
print(
    f"Total oficial (com exterior): Lula {pt_of:,} ({100 * pt_of / (pt_of + pl_of):.2f}%) x Flávio {pl_of:,} ({100 * pl_of / (pt_of + pl_of):.2f}%)"
)
print(
    f"Margem: {pt_of - pl_of:,} votos ({100 * (pt_of - pl_of) / (pt_of + pl_of):.2f} p.p. entre os finalistas)"
)
print(f"Abstenção: {100 * tb.abstencao / tb.eleitorado:.2f}% dos {int(tb.eleitorado):,} aptos")

br = virada.resumo_grupo(d.assign(_g="Brasil sem exterior"), "_g").iloc[0]
print(
    f"\nBrasil sem exterior: margem 1º turno {br.margem_t1:+.2f} p.p. -> 2º turno {br.margem_t2:+.2f} p.p. (swing {br.swing:+.2f})"
)

In [ ]:
por_uf = virada.resumo_grupo(d, "uf").sort_values("swing", ascending=False)
tabela = por_uf[
    ["uf", "margem_t1", "margem_t2", "swing", "dabst_pp", "virou_pt", "virou_pl"]
].round(2)
print("UFs vencidas por Lula no 2º turno:", int((por_uf.margem_t2 > 0).sum()), "de", len(por_uf))
print(tabela.to_string(index=False))

## 2. Swing por UF e por porte

O swing de um município pequeno oscila muito por acaso (poucos eleitores). Por isso o mapa do site trunca a escala de cor em ±10 p.p. e mostra os eleitores aptos no balão; aqui o porte aparece junto, com o número de municípios e de eleitores.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
o = por_uf.sort_values("swing")
ax.barh(o.uf.str.upper(), o.swing, color=COR_PT)
ax.axvline(0, color="#444", lw=0.8)
ax.set_xlabel("Swing da margem entre os finalistas (p.p., + = para o PT)")
ax.set_title("Swing do 1º para o 2º turno, por UF (Brasil sem exterior)")
g.salvar(fig, "virada_swing_uf")
plt.show()

In [ ]:
d["porte"] = virada.faixa_porte(d["eleitorado_t1"])
ordem = [r for r, _, _ in virada.FAIXAS_PORTE]
por_porte = virada.resumo_grupo(d, "porte").set_index("porte").loc[ordem].reset_index()
print(
    por_porte[
        ["porte", "municipios", "eleitorado", "swing", "swing_mediana_mun", "dabst_pp", "virou_pt"]
    ]
    .round(2)
    .to_string(index=False)
)

n_alem = int((d.swing.abs() > 10).sum())
print(
    f"\nMunicípios com |swing| > 10 p.p. (acima da escala truncada do mapa): {n_alem} de {len(d)}"
)
pequenos = d[d.eleitorado_t2 < 5000]
print(
    f"Municípios com menos de 5 mil aptos: {len(pequenos)}; desvio-padrão do swing neles = {pequenos.swing.std():.2f} p.p. contra {d[d.eleitorado_t2 >= 50000].swing.std():.2f} p.p. nos de 50 mil ou mais"
)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
x = range(len(por_porte))
ax.bar(x, por_porte.swing, color=COR_PT)
ax.set_xticks(list(x), por_porte.porte, rotation=25, ha="right")
ax.set_ylabel("Swing (p.p.)")
ax.set_title("Swing por porte do município (eleitorado)")
g.salvar(fig, "virada_swing_porte")
plt.show()

## 3. Quem virou

Municípios que mudaram de lado entre os turnos: o PT passou à frente em uns e o PL em outro. Minas Gerais é a única UF que trocou de lado.

In [ ]:
print("Passaram do PL para o PT:", int(d.virou_pt.sum()))
print("Passaram do PT para o PL:", int(d.virou_pl.sum()))
print()
pl = d[d.virou_pl]
print(
    pl[["uf", "nm_mun_t1", "eleitorado_t2", "margem_t1", "margem_t2", "swing"]]
    .round(2)
    .to_string(index=False)
)
print()
viraram_uf = d[d.virou_pt].groupby("uf").size().sort_values(ascending=False)
print("Municípios que viraram para o PT, por UF (maiores):")
print(viraram_uf.head(8).to_string())

In [ ]:
mg = d[d.uf == "mg"].copy()
mp = carga.mesorregiao_por_municipio("MG")  # geobr (IBGE 2018); precisa de internet na 1ª vez
mg["mesorregiao"] = mg["cd_mun_ibge_t1"].astype(int).map(mp)
assert mg["mesorregiao"].notna().all()
mgm = virada.resumo_grupo(mg, "mesorregiao").sort_values("swing")
mgm["parte_do_ganho_%"] = 100 * mgm.ganho_margem_votos / mgm.ganho_margem_votos.sum()
mgm["parte_do_eleitorado_%"] = 100 * mgm.eleitorado / mgm.eleitorado.sum()
m1 = 100 * (mg.pt_t1.sum() - mg.pl_t1.sum()) / (mg.pt_t1.sum() + mg.pl_t1.sum())
m2 = 100 * (mg.pt_t2.sum() - mg.pl_t2.sum()) / (mg.pt_t2.sum() + mg.pl_t2.sum())
print(
    f"Minas Gerais: margem {m1:+.2f} -> {m2:+.2f} p.p.; {int(mg.virou_pt.sum())} municípios viraram para o PT; ganho líquido de margem {int(mg.dmargem_votos.sum()):,} votos"
)
print(
    mgm[
        [
            "mesorregiao",
            "municipios",
            "margem_t1",
            "margem_t2",
            "swing",
            "parte_do_ganho_%",
            "parte_do_eleitorado_%",
        ]
    ]
    .round(1)
    .to_string(index=False)
)
print(
    "\nDescrição da distribuição (sem inferência de causa); mesorregiões = divisão do IBGE de 2018."
)

## 4. Padrão do swing (descritivo)

O swing foi maior onde o PT tinha menos votos no 1º turno. **Isto é um padrão descritivo, não uma explicação**: parte pode ser regressão à média (margens extremas tendem a se mover mais por ruído) e os dados são agregados. Para não ser só diferença entre estados, o modelo abaixo tem efeito fixo de UF.

In [ ]:
r = virada.regressao_swing(d)
print(f"municípios: {int(r['n'])}")
print(f"correlação (logit do PT no 1º turno × swing em logit): {r['correlacao']:+.3f}")
print(
    f"inclinação dentro da UF (efeito fixo de UF, peso = votos dos finalistas no 2º turno): {r['inclinacao_dentro_da_uf']:+.3f}"
)
print(f"R² ponderado do modelo completo: {r['r2_ponderado']:.3f}")

In [ ]:
s = virada.swing_logit(d)
fig, ax = plt.subplots(figsize=(7, 5))
hb = ax.hexbin(s.x, s.sl, gridsize=45, bins="log", cmap="Blues", mincnt=1)
ax.axhline(0, color="#444", lw=0.8)
ax.set_xlabel("PT no 1º turno, entre os finalistas (logit)")
ax.set_ylabel("Swing em logit (+ = para o PT)")
ax.set_title("Swing × força do PT no 1º turno (um ponto = um município)")
fig.colorbar(hb, label="municípios (escala log)")
g.salvar(fig, "virada_padrao_swing")
plt.show()

## 5. Fluxo do 1º para o 2º turno (só o que é robusto)

Matriz ecológica com 8 origens (6 grupos de candidatos do 1º turno, brancos/nulos e abstenção) e 4 destinos (PL, PT, brancos/nulos, abstenção), estimada por Goodman restrito (B ≥ 0, linhas somam 1). O intervalo trata **cada UF inteira como unidade** (o mais conservador; o bootstrap por município dá intervalos falsamente estreitos).

**A matriz completa fica fora.** As linhas dos terceiros colocados variam muito de região para região e as parcelas de origem são quase colineares; só entram resultados que sobrevivem a isso.

In [ ]:
cfg = pj.carregar_config_projecao(config.RAIZ / "config" / "projecao_t2.yaml")
grupos = ie.grupos_t1_2026(cfg, "base")
origens, c1, x, y, w = fluxo.montar_t1_t2(
    carga.candidatos_2026(),
    carga.totais_2026(),
    carga.candidatos_t2_2026(),
    carga.totais_t2_2026(),
    grupos,
    pj.CAT_BN,
    pj.CAT_ABST,
)
uf = c1["uf"].to_numpy()
b = fluxo.estimar_b(x, y, w)
N_BOOT = 300  # no relatório do site foram 1.000; aqui 300 para o notebook rodar em poucos minutos
boot = fluxo.bootstrap(lambda r: fluxo.indices_cluster_uf(uf, r), x, y, w, N_BOOT, semente=0)
lo, hi = ie.intervalo_percentil(boot)

cond = fluxo.condicionamento(x, w)
print(f"{len(w)} municípios, {w.sum():,.0f} aptos; origens: {origens}")
print(
    f"condicionamento das parcelas de origem: número de condição {cond['numero_condicao']:,.0f}; maior correlação entre pares {cond['maior_correlacao_abs']:.2f}"
)

In [ ]:
def linha(o, dest):
    a, j = origens.index(o), ie.COLS_T2.index(dest)
    return f"{100 * b[a, j]:5.1f}%  (IC por UF {100 * lo[a, j]:.1f}% a {100 * hi[a, j]:.1f}%)"


robustos = [
    ("flavio", "PL"),
    ("lula", "PT"),
    ("abst_2026", "ABST"),
    ("bn_2026", "PT"),
    ("bn_2026", "BN"),
]
rot = {
    "flavio": "Eleitores de Flávio",
    "lula": "Eleitores de Lula",
    "abst_2026": "Abstenção do 1º turno",
    "bn_2026": "Brancos e nulos do 1º turno",
}
for o, dest in robustos:
    print(f"{rot[o]:30s} -> {dest:4s} {linha(o, dest)}")

In [ ]:
# Piso aritmético, sem modelo: quantos votos do PT vieram de quem NÃO votou em Lula no 1º turno
aptos_t1 = c1[origens].sum(axis=1).to_numpy()
gm = fluxo.ganhos_minimos_pt(x, y, aptos_t1, col_lula=origens.index("lula"))
pt_t2 = (y[:, 1] * w).sum()
lula_t1 = (x[:, origens.index("lula")] * w).sum()
print(f"PT no 2º turno (Brasil sem exterior): {pt_t2:,.0f} votos; Lula no 1º turno: {lula_t1:,.0f}")
print(
    f"Piso aritmético por município: pelo menos {gm['ganho_minimo_votos']:,.0f} votos do PT vieram de quem não votou em Lula no 1º turno"
)

In [ ]:
# Por que a matriz completa fica fora: as linhas dos terceiros mudam muito entre regiões
reg = c1["uf"].map(regioes.regiao_da_uf).to_numpy()
linhas = []
for r_ in ["N", "NE", "CO", "SE", "S"]:
    m = reg == r_
    bb = fluxo.estimar_b(x[m], y[m], w[m])
    linhas.append(
        {
            "região": r_,
            **{f"{o}->PT": bb[origens.index(o), 1] for o in ("cury", "renan", "caiado", "bn_2026")},
        }
    )
het = pd.DataFrame(linhas)
print("Parcela que foi para o PT, estimada por região:")
print((100 * het.set_index("região")).round(0).to_string())
print(
    "\nCaiado->PT vai de ~0% a ~70% e Renan->PT de ~10% a ~90% conforme a região: o ponto único nacional não é publicável. Só brancos/nulos -> PT é majoritário em todas."
)

## 6. Extensão do horário de votação: a aritmética da margem

A votação foi estendida até as 19h por liminar do STF (ministro Zanin), assinada logo depois das 17h; as fontes divergem sobre o alcance (todo o país × só as seções ainda abertas). O número de votos depois das 17h abaixo é **da imprensa** (levantamento do g1 com os logs das urnas, reproduzido por outros veículos) e **não foi verificado** nos dados deste projeto: os arquivos usados aqui não têm horário por voto.

In [ ]:
margem = pt_of - pl_of
apos = VOTOS_APOS_17H_IMPRENSA
print(f"Margem oficial: {margem:,} votos")
print(
    f"Votos depois das 17h (imprensa, NÃO verificado aqui): {apos:,} = {100 * apos / margem:.2f}% da margem; a margem é {margem / apos:.0f} vezes maior"
)
print(
    f"Mesmo que todos fossem de Lula e fossem descartados, a margem seria de {margem - apos:,} votos"
)

nao_inst = int(tb["extra_s_sni"]), int(tb["extra_e_esni"])
print(
    f"\nSeções não instaladas: {nao_inst[0]} (todas no exterior), com {nao_inst[1]} eleitores aptos: sem efeito sobre a margem"
)

**Pendente**: o cruzamento de comparecimento e voto nas seções com fechamento estendido depende de dados por seção com horário; o dataset `votacao_secao_2026` do Dados Abertos não confirma que traz o 2º turno. Quando existir, o cruzamento descreve diferenças e **não decide a validade da alegação**: a aritmética acima já fecha a pergunta sobre o resultado.

## Resumo para quem chega agora
- Lula ganhou por 4.430.191 votos; o 1º turno tinha Flávio 1,87 p.p. à frente.
- O movimento foi para o PT em todo o país, maior nas cidades grandes e onde o PT tinha menos votos; 211 municípios (76 em Minas) passaram para o PT e 1 (Uirapuru/GO) para o PL.
- Do fluxo, só se sustenta: quem votou em Flávio ficou com ele, quem votou em Lula ficou com ele, a abstenção continuou abstenção e os brancos/nulos foram majoritariamente para o PT; pelo menos 10,35 milhões dos votos do PT vieram de quem não votou nele no 1º turno.